(sup_reg_ex: develop)=
# Regression Model Development (part 1)

Supervised algorithms use inputs (independent variables) and labeled outputs (dependent variable -the "answers") to create a model that can measure its performance and learn over time. Splitting the data into independent and dependent variables, we have the following (again, this will be very similar to the [previous example](sup_class_ex:develop)):

In [1]:
#Note: we only repeat this step from before, because this is a new .ipyb page.
#   it only needs to be executed once per file.

#We'll import libraries as needed, but when submitting, having them all at the top is best practice
import pandas as pd

# Reloading the dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv"
df = pd.read_csv(url) #read CSV into Python as a dataframe

column_names = ['sepal-length', 'sepal-width', 'petal-length', 'petal-width', 'type']
df = pd.read_csv(url, names = column_names) #read CSV into Python as a dataframe

#Choosing the variables. 
X = df.drop(columns=['sepal-length']) #indpendent variables
y = df[['sepal-length']].copy() #dependent variables

(sup_reg_ex: develop: train)=
## Train Model

Recall, splitting the data into training and testing sets is not required, but it is good practice. Furthermore, it provides content for part D. As with the [previous example](sup_class_ex:develop), we'll use [scikit-learn aka sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) built-ins for this.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

#split the variable sets into training and testing subsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.333, random_state=41)

In [3]:
#Nice displays are nice but not required. 
from IPython.display import display_html 
X_train_styler = X_train.head(5).style.set_table_attributes("style='display:inline'").set_caption('Independents variables')
y_train_styler = y_train.head(5).style.set_table_attributes("style='display:inline'").set_caption('Dependents variables')
space = "\xa0" * 10 #space between columns
display_html(X_train_styler._repr_html_()+ space  + y_train_styler._repr_html_(), raw=True)

Independents variables 
 
 
   
 sepal-width 
 petal-length 
 petal-width 
 type 
 
 
 
 
 111 
 2.700000 
 5.300000 
 1.900000 
 Iris-virginica 
 
 
 82 
 2.700000 
 3.900000 
 1.200000 
 Iris-versicolor 
 
 
 130 
 2.800000 
 6.100000 
 1.900000 
 Iris-virginica 
 
 
 27 
 3.500000 
 1.500000 
 0.200000 
 Iris-setosa 
 
 
 33 
 4.200000 
 1.400000 
 0.200000 
 Iris-setosa 
 
 
 
          
 
 Dependents variables 
 
 
   
 sepal-length 
 
 
 
 
 111 
 6.400000 
 
 
 82 
 5.800000 
 
 
 130 
 7.400000 
 
 
 27 
 5.200000 
 
 
 33 
 5.500000


Review sklearn's nice [supervised learning library](https://scikit-learn.org/stable/supervised_learning.html). Note that many of these models have both classification and regression extensions.

In [4]:
from sklearn.linear_model import LinearRegression 

Our data is mostly quantitative and the scatterplots indicate some linear relations between variables. So linear regression isn't a bad place to start. Once we've trained and tested a linear regression model, we'll easily be able to experiment with different algorithms. 

:::{margin} Is linear regression ML?
It depends on who you ask. Google "Is linear regression machine learning?" and you'll see some interesting (and entertaining) discussion. For the capstone, it applies an algorithm to data so the answer is -yes.  
:::

In [5]:
linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train,y_train)

ValueError: could not convert string to float: 'Iris-virginica'

*Error? Wait what happened!?!* This error returns a lot of output, but the last line makes things clear:

`ValueError: could not convert string to float: 'Iris-virginica'`

The algorithm expected numbers; it does not know what to do with the flower types (strings). So how do we fix this?

(sup_reg_ex: develop: train: categorical_1)=
## Processing Categorical Data (the easy way)

One way to fix a problem is to avoid it. You are not required to use all the data -only some of it. Sometimes choosing the right variables is the real trick. [Dimensionality reduction](https://en.wikipedia.org/wiki/Dimensionality_reduction) is an important part of the data sciences. Here, those flower types DO matter, and it would be best to include that data -but goal #1 is to get things working. Improving things is step #2 and step #3 and step #4 and ... step $\# \infty$.

So just to get things rolling, let's remove the column with the categorical data:

In [6]:
X_train_no_type = X_train.drop(columns = ['type'])
X_test_no_type =  X_test.drop(columns = ['type'])

X_test_no_type

,sepal-width,petal-length,petal-width
119,2.2,5.0,1.5
128,2.8,5.6,2.1
135,3.0,6.1,2.3
91,3.0,4.6,1.4
112,3.0,5.5,2.1
71,2.8,4.0,1.3
123,2.7,4.9,1.8
85,3.4,4.5,1.6
147,3.0,5.2,2.0
143,3.2,5.9,2.3


Now the models will train without error:

In [7]:
linear_reg_model.fit(X_train_no_type, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1, 3)","[[ 0.7 , 0.81,-0.79]]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['sepal-width','petal-length','petal-width']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](1,)",[1.57]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,3


And the model can make predictions for an entire set:

In [8]:
y_pred_no_type = linear_reg_model.predict(X_test_no_type)
y_pred_no_type

array([[5.98770812],
       [6.42220879],
       [6.81026327],
       [6.3038615 ],
       [6.48153317],
       [5.75587752],
       [6.02106191],
       [6.34587216],
       [6.31716812],
       [6.788514  ],
       [6.23165894],
       [5.01764515],
       [4.57470184],
       [5.07393461],
       [4.87302581],
       [6.48997581],
       [4.88812177],
       [6.34092093],
       [4.885904  ],
       [4.00445291],
       [5.46842817],
       [6.62636673],
       [4.85349432],
       [4.71509986],
       [5.50178197],
       [5.57125107],
       [6.43782043],
       [6.00331976],
       [4.86637251],
       [6.31473613],
       [5.44667891],
       [6.08460761],
       [5.63522521],
       [6.01862993],
       [6.08060051],
       [5.19561829],
       [6.06972588],
       [6.28433001],
       [6.47065854],
       [6.29098332],
       [6.06929744],
       [6.16124571],
       [5.58789409],
       [6.64589822],
       [6.60683523],
       [5.3817326 ],
       [6.61032665],
       [4.892

Or a single input:

In [9]:
# The model was trained with a dataframe, so you can only predict on dataframes
# Recall we removed the petal type, and we are predicting the sepal-length
column_names_short = ['sepal-width', 'petal-length', 'petal-width']

# Creates a dataframe from a single element for input. This avoids a warning for missing feature names. 
#Alternatively, you can use print(linear_reg_model.predict([[3.2, 1.3, .2]])) without error. 
input_df = pd.DataFrame(np.array([[3.2, 1.3, .2]]), columns = column_names_short)

print(linear_reg_model.predict(input_df))

[[4.71509986]]


```{note}
Your model can only predict on data simliar to what it was trained with. Since this model was trained with a dataframe, a matching new dataframe, 'input_df' was created to predict a single input. Alternatively, we could have converted the original data to an array using `ravel()` or `.values`  (see the [previous example](sup_class_ex:develop:train)).                       
```

(sup_reg_ex: develop: accuracy)=
## Accuracy Analysis (for Regression) part 1

Now that the model works, we can work on improving it. But first we'll need metrics so we can tell if we're making progress. As we are trying to predict a continuous number, even the very best model will have errors in almost every prediction (if not, it's almost certainly [overfitted](https://en.wikipedia.org/wiki/Overfitting)). Whereas measuring the success of our classification model was a simple ratio, here we need a way to measure how much those predictions deviate from actual values. See sklearn's list of [metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) for regression. 

To get things started, we'll use the *mean squared error* (MSE), a popular metric for evaluating regression models. Regression metrics are covered in more depth in the [Regression Accuracy Analysis](sup_reg_ex: develop: accuracy) section. 

In [10]:
from sklearn.metrics import mean_squared_error

mean_squared_error(y_test, y_pred_no_type)

0.12264182555541722

What does this mean? The closer the MSE is to 0, the better. However, this value is not given in terms of the dependent variable, and MSE values are not comparable across use cases, i.e., comparing an MSE from your project to that of a different model is not comparing "apples to apples." A "good" MSE will depend on your data and project needs.

This value can be used to determine if tweaks improve the results. For example, reviewing the [visualizations of this dataset](sup_class_ex:descriptive_methods_and_visualizations), we might expect that the regression coefficients (numbers determining the lines directions) should be positive, and try `LinearRegression(positive = True)`.

In [11]:
linear_reg_model2 = LinearRegression(positive = True)
linear_reg_model2.fit(X_train_no_type, y_train)
y_pred_no_type = linear_reg_model2.predict(X_test_no_type)

mean_squared_error(y_test, y_pred_no_type)

0.10302108975537984

And $0.103 < 0.122$.